# 01 — Découvrir les événements et les résolutions V2

Objectif : suivre un événement brut jusqu'au marché, puis distinguer une issue officielle d'un simple prix observé.

In [ ]:
import polars as pl

from poly_data.io.parquet_store import ParquetStore
from poly_data.notebooks import resolve_notebook_context, source_inventory

ctx = resolve_notebook_context()
store = ParquetStore(ctx.root)
SOURCES = ['order_filled_v2', 'trades', 'markets_current', 'market_assets', 'market_outcomes']
missing = [source for source in SOURCES if not list((ctx.root / source).rglob('*.parquet'))]
if missing:
    raise FileNotFoundError(
        f'Missing sources: {missing}. Build the fixture with: '
        'uv run python scripts/make_synthetic_smoke_fixture.py'
    )
print({'root': str(ctx.root), 'mode': ctx.mode, 'revision': ctx.revision})
source_inventory(store, SOURCES)

## Provenance d'un événement

Chaque ligne brute a un bloc, une transaction, un ordre et un actif. L'étape silver ne devine pas son marché : elle passe par la dimension ``market_assets``.

In [ ]:
raw = store.scan('order_filled_v2')
raw.select([
    pl.len().alias('events'),
    pl.col('block_number').min().alias('first_block'),
    pl.col('block_number').max().alias('last_block'),
    pl.col('asset').n_unique().alias('assets'),
]).collect()

In [ ]:
events = store.scan('order_filled_v2').select(['id', 'timestamp', 'asset', 'price', 'side'])
assets = store.scan('market_assets').select(['asset', 'market_id', 'token_side'])
events.join(assets, on='asset', how='left').head(10).collect()

## Résultats officiels versus proxys de prix

Un dernier prix proche de 0 ou 1 peut être intéressant à explorer, mais ce n'est pas une résolution. Les métriques supervisées et les PnL réalisés ci-dessous utilisent uniquement ``market_outcomes``. Les marchés sans ligne dans cette table restent censurés.

In [ ]:
trades = store.scan('trades').select(['market_id', 'timestamp', 'price', 'nonusdc_side'])
official = store.scan('market_outcomes').select(['market_id', 'winner_token', 'resolved_at'])
last_prices = (
    trades.sort('timestamp').group_by('market_id').last()
    .select(['market_id', pl.col('price').alias('last_observed_price')])
)
(
    official.join(last_prices, on='market_id', how='left')
    .join(store.scan('markets_current').select(['id', 'question']), left_on='market_id', right_on='id')
    .sort('resolved_at').head(10).collect()
)

La colonne ``winner_token`` est la référence finale. ``last_observed_price`` reste un diagnostic : elle ne doit pas être transformée en étiquette pour évaluer un modèle.